# A November dry run check

## **Reads data/dryrun, deleted after the run; outputs kept as the record**

After the dry run was performed, we want to compare the backtest results with the November dry run

In [11]:
# Imports
import duckdb
import pandas as pd
from vportfolio import allocators, data, engine, pipelines
from vportfolio.params import LIVE_PORTFOLIOS

In [12]:
# the live record, from the dry-run store (read-only)
con = duckdb.connect("../data/dryrun/portfolio.duckdb", read_only = True)
live_nav = con.execute("SELECT date, portfolio, nav FROM nav").df()
con.close()
live_nav["date"] = pd.to_datetime(live_nav["date"])
live_nav = live_nav.pivot(index = "date", columns = "portfolio", values = "nav")

In [13]:
# the backtest over the same days, from today's download
raw = pd.read_parquet("../data/dryrun/raw/yahoo_prices_2026-11-20.parquet")
processed = pipelines.data_pipeline(raw)
bench = data.benchmark_data(raw)
period = pd.Series(pd.to_datetime(["2026-10-30", "2026-11-18"])) # trades 2 Nov, holds to 18 Nov

In [14]:
# compare the two
for name, settings in LIVE_PORTFOLIOS.items():
    pdata = bench if settings["allocator"] == "sixty_forty" else processed
    # research engine over 2-18 of November
    result = engine.backtest_engine(getattr(allocators, settings["allocator"]),
                                    pdata["close"], pdata["open"], pdata["returns"],
                                    pdata["eligible"], period, processed["cost_bp"],
                                    settings["cap"], settings["lookback"])
    # compare the two series side by side
    both = pd.concat({"backtest": result["NAV"], "live": live_nav[name]}, axis = 1).dropna()
    # print the difference
    print(name, len(both), "days, largest difference:", (both["backtest"] - both["live"]).abs().max())

nco 13 days, largest difference: 2.6670376018778086e-08
equal_weights 13 days, largest difference: 2.220446049250313e-16
sixty_forty 13 days, largest difference: 4.440892098500626e-16


The data are sufficiently close so the dry run is succesfull. I also checked the log in the terminal and then intentionally run a failed dry run and check the log again to make sure everything is in order. Everything went succesfully